In [5]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# 配置区（按需修改）
# =========================
RECO_CSV = "results/10_sgd_linear_regression.csv"
OUT_DIR = "Backtest Results"
os.makedirs(OUT_DIR, exist_ok=True)

# 同时回测多个K
TOPK_LIST = [5, 10, 20]
DEFAULT_K = 3

TRADING_DAYS_PER_YEAR = 252

# true_open_return 如果是 6.6 表示 6.6%，则 True
RETURN_IS_PERCENT = True

# =========================
# 交易成本设定（按你最终确认的口径）
# 单位：bps（1 bps = 万1 = 0.01%）
# =========================
COMMISSION_BPS_ONE_WAY = 2.0     # 佣金，买卖双边
STAMP_TAX_BPS_SELL = 5.0         # 印花税，仅卖出
TRANSFER_FEE_BPS_ONE_WAY = 0.1   # 过户费，买卖双边
SLIPPAGE_BPS_ONE_WAY = 2.0       # 滑点，买卖双边

# =========================
# 指标计算函数
# =========================
def max_drawdown(equity: np.ndarray) -> float:
    """
    返回最大回撤（负数）
    """
    if equity.size == 0:
        return 0.0
    peak = np.maximum.accumulate(equity)
    dd = equity / np.maximum(peak, 1e-12) - 1.0
    return float(dd.min())

def annualized_return(daily_ret: np.ndarray, n_days: int, trading_days=252) -> float:
    """
    基于完整测试期日收益序列计算年化收益率
    """
    if n_days <= 0:
        return 0.0
    equity = np.cumprod(1.0 + daily_ret)
    total = equity[-1] - 1.0
    return float((1.0 + total) ** (trading_days / n_days) - 1.0)

def annualized_vol(daily_ret: np.ndarray, trading_days=252) -> float:
    """
    年化波动率
    """
    if daily_ret.size <= 1:
        return 0.0
    return float(np.std(daily_ret, ddof=1) * math.sqrt(trading_days))

def sharpe_ratio(daily_ret: np.ndarray, trading_days=252, rf=0.0) -> float:
    """
    夏普比率（默认无风险利率 rf=0）
    """
    if daily_ret.size <= 1:
        return 0.0
    mu = np.mean(daily_ret) - rf / trading_days
    sd = np.std(daily_ret, ddof=1)
    return float(mu / (sd + 1e-12) * math.sqrt(trading_days))

def summarize_backtest(df_daily: pd.DataFrame) -> dict:
    """
    指标口径：
    - TotalReturn / AnnReturn / MDD / Sharpe / Calmar / AnnVol
      基于完整测试期（空仓日收益记0）
    - WinRate
      仅按实际交易日计算
    """
    r = df_daily["net_ret"].values.astype(float)
    n = len(r)
    eq = df_daily["equity"].values.astype(float)

    # 最大回撤
    mdd = max_drawdown(eq)
    # 年化收益率
    ann_ret = annualized_return(r, n, TRADING_DAYS_PER_YEAR)
    # 年化波动率
    ann_vol = annualized_vol(r, TRADING_DAYS_PER_YEAR)
    # 夏普比率
    sharpe = sharpe_ratio(r, TRADING_DAYS_PER_YEAR, rf=0.0)
    # 卡玛比率
    calmar = float(ann_ret / (abs(mdd) + 1e-12))

    trade_mask = df_daily["n_hold"].values > 0
    if trade_mask.sum() > 0:
        # 胜率
        win_rate = float((df_daily.loc[trade_mask, "net_ret"] > 0).mean())
        # 平均收益
        avg_trade_ret = float(df_daily.loc[trade_mask, "net_ret"].mean())
    else:
        win_rate = 0.0
        avg_trade_ret = 0.0

    return {
        "Days": int(n),  # 完整测试期天数
        "TradeDays": int(trade_mask.sum()),  # 实际交易日数
        "NoTradeDays": int((~trade_mask).sum()),  # 空仓日数
        "FinalEquity": float(eq[-1]) if n > 0 else 1.0,
        "TotalReturn": float(eq[-1] - 1.0) if n > 0 else 0.0,
        "AnnReturn": ann_ret,
        "AnnVol": ann_vol,
        "Sharpe": sharpe,
        "MaxDrawdown": mdd,
        "Calmar": calmar,
        "WinRate": win_rate,
        "AvgDailyRet": float(np.mean(r)) if n > 0 else 0.0,
        "StdDailyRet": float(np.std(r, ddof=1)) if n > 1 else 0.0,
        "AvgTradeDayRet": avg_trade_ret,
    }

# =========================
# 数据读取
# =========================
def load_reco(path: str):
    """
    返回：
    - df_all: 原始推荐数据（做基础清洗）
    - all_dates: 原始完整测试期日期序列（用于保留空仓日）
    """
    df = pd.read_csv(path)

    need_cols = {"date", "rank", "code", "score", "true_open_return"}

    miss = need_cols - set(df.columns)
    if miss:
        raise ValueError(f"reco file missing columns: {miss}")

    df["date"] = df["date"].astype(str)
    df["code"] = df["code"].astype(str)
    df["rank"] = df["rank"].astype(int)
    df["score"] = pd.to_numeric(df["score"], errors="coerce")
    df["true_open_return"] = pd.to_numeric(df["true_open_return"], errors="coerce")

    # 原始完整日期序列
    all_dates = sorted(df["date"].dropna().unique().tolist())

    # 真实隔夜收益： (Open_{t+1} - Close_t) / Close_t
    ret = df["true_open_return"].astype(float).values
    if RETURN_IS_PERCENT:
        ret = ret / 100.0
    df["ret_gap"] = ret

    df_all = df.copy()

    return df_all, all_dates

# =========================
# 成本函数
# =========================
def get_cost_components():
    """
    返回买入端、卖出端、双边总成本（小数形式）
    """
    buy_cost = (
        COMMISSION_BPS_ONE_WAY
        + TRANSFER_FEE_BPS_ONE_WAY
        + SLIPPAGE_BPS_ONE_WAY
    ) / 10000.0

    sell_cost = (
        COMMISSION_BPS_ONE_WAY
        + TRANSFER_FEE_BPS_ONE_WAY
        + SLIPPAGE_BPS_ONE_WAY
        + STAMP_TAX_BPS_SELL
    ) / 10000.0

    roundtrip_cost = buy_cost + sell_cost
    return buy_cost, sell_cost, roundtrip_cost

# =========================
# 回测主逻辑
# =========================
def backtest_for_k(df_all: pd.DataFrame, all_dates: list, k: int):
    """
    回测逻辑：
    - 在每个日期内取 rank <= k 的股票
    - 对剩余股票等权买入
    - 若某日剩余股票数为0，则该日为空仓日，收益记0
    """
    buy_cost, sell_cost, roundtrip_cost = get_cost_components()

    df_k = df_all[df_all["rank"] <= k].copy()

    # 按日期分组，便于快速查找
    grouped = {d: sub.copy() for d, sub in df_k.groupby("date", sort=True)}

    daily_rows = []
    trade_rows = []

    for d in all_dates:
        sub = grouped.get(d, None)

        if sub is None or len(sub) == 0:
            # 空仓日：收益记0，净值不变
            daily_rows.append({
                "date": d,
                "K": k,
                "n_hold": 0,
                "gross_ret": 0.0,
                "net_ret": 0.0,
                "buy_cost": 0.0,
                "sell_cost": 0.0,
                "roundtrip_cost": 0.0,
                "is_trade_day": 0,
                "trade_rule": "Buy@Close(t) Sell@Open(t+1) | EqualWeightOnRemaining"
            })
            continue

        # 对筛选后剩余股票等权
        rets = sub["ret_gap"].values.astype(float)
        n_hold = len(rets)

        gross = float(np.mean(rets))
        net = gross - roundtrip_cost

        daily_rows.append({
            "date": d,
            "K": k,
            "n_hold": int(n_hold),
            "gross_ret": gross,
            "net_ret": net,
            "buy_cost": buy_cost,
            "sell_cost": sell_cost,
            "roundtrip_cost": roundtrip_cost,
            "is_trade_day": 1,
            "trade_rule": "Buy@Close(t) Sell@Open(t+1) | EqualWeightOnRemaining"
        })

        # 记录个股明细
        for _, r in sub.iterrows():
            trade_rows.append({
                "date": d,
                "K": k,
                "code": r["code"],
                "rank": int(r["rank"]),
                "score": float(r["score"]),
                "ret_gross": float(r["ret_gap"]),
                "buy_cost": buy_cost,
                "sell_cost": sell_cost,
                "ret_net": float(r["ret_gap"] - roundtrip_cost),
                "trade_rule": "Buy@Close(t) Sell@Open(t+1)"
            })

    df_daily = pd.DataFrame(daily_rows).sort_values("date").reset_index(drop=True)
    df_daily["equity"] = (1.0 + df_daily["net_ret"]).cumprod()

    df_trades = pd.DataFrame(trade_rows)
    if not df_trades.empty:
        df_trades = df_trades.sort_values(["date", "rank"]).reset_index(drop=True)

    return df_daily, df_trades

def format_summary_for_display(df_sum: pd.DataFrame) -> pd.DataFrame:
    """
    仅用于打印展示：把收益率转成百分比字符串
    """
    out = df_sum.copy()

    pct_cols = [
        "TotalReturn", "AnnReturn", "AnnVol", "MaxDrawdown",
        "AvgDailyRet", "StdDailyRet", "AvgTradeDayRet"
    ]
    for c in pct_cols:
        if c in out.columns:
            out[c] = out[c].map(lambda x: f"{x:.4%}")

    ratio_cols = ["Sharpe", "Calmar", "WinRate", "FinalEquity", "AvgHold"]
    for c in ratio_cols:
        if c in out.columns:
            if c == "WinRate":
                out[c] = out[c].map(lambda x: f"{x:.2%}")
            elif c == "FinalEquity":
                out[c] = out[c].map(lambda x: f"{x:.4f}")
            else:
                out[c] = out[c].map(lambda x: f"{x:.4f}")

    return out

def run_all():
    print("Loading data...")
    df_all, all_dates = load_reco(RECO_CSV)

    print(f"[Info] Total test dates: {len(all_dates)}")
    print(f"[Info] Total records: {len(df_all)}")

    buy_cost, sell_cost, roundtrip_cost = get_cost_components()
    print("\n========== Cost Setting ==========")
    print(f"Buy side cost    = {buy_cost:.6f} ({buy_cost*100:.4f}%)")
    print(f"Sell side cost   = {sell_cost:.6f} ({sell_cost*100:.4f}%)")
    print(f"Roundtrip cost   = {roundtrip_cost:.6f} ({roundtrip_cost*100:.4f}%)")
    print("  = Commission + TransferFee + Slippage (buy)")
    print("  + Commission + TransferFee + Slippage + StampTax (sell)")

    summary_list = []
    daily_map = {}

    for k in TOPK_LIST:
        print(f"\nBacktesting Top-{k}...")
        df_daily, df_trades = backtest_for_k(
            df_all=df_all,
            all_dates=all_dates,
            k=k
        )

        # 保存明细
        df_daily.to_csv(os.path.join(OUT_DIR, f"10_sgd_linear_regression_bt_daily_K{k}.csv"), index=False)
        df_trades.to_csv(os.path.join(OUT_DIR, f"10_sgd_linear_regression_bt_trades_K{k}.csv"), index=False)

        summ = summarize_backtest(df_daily)
        summ.update({
            "K": k,
            "CommissionBpsOneWay": COMMISSION_BPS_ONE_WAY,
            "StampTaxBpsSell": STAMP_TAX_BPS_SELL,
            "TransferFeeBpsOneWay": TRANSFER_FEE_BPS_ONE_WAY,
            "SlippageBpsOneWay": SLIPPAGE_BPS_ONE_WAY,
            "AvgHold": float(df_daily["n_hold"].replace(0, np.nan).mean()) if (df_daily["n_hold"] > 0).any() else 0.0,
            "TradeRule": "Buy@Close(t) Sell@Open(t+1) | EqualWeightOnRemaining | KeepNoTradeDays"
        })
        summary_list.append(summ)
        daily_map[k] = df_daily

    df_sum = pd.DataFrame(summary_list).sort_values("K").reset_index(drop=True)
    df_sum.to_csv(os.path.join(OUT_DIR, "10_sgd_linear_regression_bt_summary.csv"), index=False)

    print("\n========== Backtest Summary ==========")
    print(format_summary_for_display(df_sum).to_string(index=False))

    # 净值曲线
    plt.figure(figsize=(10, 5))
    for k in TOPK_LIST:
        if k in daily_map:
            d = daily_map[k]
            plt.plot(pd.to_datetime(d["date"]), d["equity"], label=f"Top-{k}")
    plt.title("Equity Curve (Net) | Buy@Close(t) Sell@Open(t+1)")
    plt.xlabel("Date")
    plt.ylabel("Equity")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "10_sgd_linear_regression_equity_curve.png"), dpi=200)
    plt.close()
    
if __name__ == "__main__":
    run_all()

Loading data...
[Info] Total test dates: 205
[Info] Total records: 4100

========== Cost Setting ==========
Buy side cost    = 0.000410 (0.0410%)
Sell side cost   = 0.000910 (0.0910%)
Roundtrip cost   = 0.001320 (0.1320%)
  = Commission + TransferFee + Slippage (buy)
  + Commission + TransferFee + Slippage + StampTax (sell)

Backtesting Top-5...

Backtesting Top-10...

Backtesting Top-20...

========== Backtest Summary ==========
 Days  TradeDays  NoTradeDays FinalEquity TotalReturn AnnReturn   AnnVol  Sharpe MaxDrawdown  Calmar WinRate AvgDailyRet StdDailyRet AvgTradeDayRet  K  CommissionBpsOneWay  StampTaxBpsSell  TransferFeeBpsOneWay  SlippageBpsOneWay AvgHold                                                              TradeRule
  205        205            0      0.6590   -34.0964% -40.1050% 13.0344% -3.8631   -35.7825% -1.1208  29.27%    -0.1998%     0.8211%       -0.1998%  5                  2.0              5.0                   0.1                2.0  5.0000 Buy@Close(t) Sell@O